In [ ]:
# In[1]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[1]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[2]:

import pandas as pd

# Load CSVs into DataFrames and parse timestamps as UTC
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# 1) metric.csv summary
metric_cmdb_ids = pd.Series(pd.unique(metric_df['cmdb_id']))[:50]
metric_kpi_names = pd.Series(pd.unique(metric_df['kpi_name']))[:50]

metric_grp = metric_df.groupby(['cmdb_id', 'kpi_name'])['value']
metric_summary_df = (
    metric_grp
    .agg(count='count', mean='mean', p95=lambda x: x.quantile(0.95))
    .reset_index()
    .sort_values('count', ascending=False)
    .head(20)
)

# 2) trace.csv summary
trace_cmdb_ids = pd.Series(pd.unique(trace_df['cmdb_id']))[:50]
trace_names = pd.Series(pd.unique(trace_df['trace_name']))[:50]

trace_grp = trace_df.groupby(['cmdb_id', 'trace_name'])['value']
trace_summary_df = (
    trace_grp
    .agg(count='count', mean='mean', p95=lambda x: x.quantile(0.95))
    .reset_index()
    .sort_values('count', ascending=False)
    .head(20)
)

# 3) log.csv summary
log_cmdb_ids = pd.Series(pd.unique(log_df['cmdb_id']))[:50]
log_names = pd.Series(pd.unique(log_df['log_name']))[:50]

log_grp = log_df.groupby(['cmdb_id', 'log_name'])['value']
log_summary_df = (
    log_grp
    .agg(count='count', mean='mean', p95=lambda x: x.quantile(0.95))
    .reset_index()
    .sort_values('count', ascending=False)
    .head(20)
)

# 4) error_logs.csv summary
# Ensure 'message' column exists
if 'message' not in error_logs_df.columns:
    error_logs_df['message'] = ''

error_cmdb_ids = pd.Series(pd.unique(error_logs_df['cmdb_id']))[:50]
error_counts_df = (
    error_logs_df['cmdb_id']
    .value_counts()
    .rename_axis('cmdb_id')
    .reset_index(name='count')
)

# Top 10 distinct message samples with counts, truncating messages to 200 chars
msg_counts = (
    error_logs_df['message']
    .fillna('')
    .astype(str)
    .value_counts()
    .reset_index()
)
# After reset_index columns are ['index', 0]; rename appropriately
msg_counts = msg_counts.rename(columns={'index': 'message', 0: 'count'})
msg_counts['message'] = msg_counts['message'].str.slice(0, 200)
error_message_samples_df = msg_counts.head(10)

# Return compact summaries (do not display full raw DataFrames)
metric_cmdb_ids, metric_kpi_names, metric_summary_df, trace_cmdb_ids, trace_names, trace_summary_df, log_cmdb_ids, log_names, log_summary_df, error_cmdb_ids, error_counts_df, error_message_samples_df

```
Out[2]:
```
summary = (
    "Summary of telemetry load results:\n"
    "- metric.csv: 50 unique cmdb_id values, 8 unique kpi_name values. Top (cmdb_id,kpi) pairs "
    "have 25 points each. Notable values: ts-admin-basic-info-service memory mean ≈236,405,540 (p95 ≈237,074,582); "
    "ts-travel2-service memory mean ≈279,992,040 (p95 ≈280,196,369); ts-travel2-service socket mean ≈19.69 (p95 ≈21.71); "
    "workload for ts-travel2-service mean ≈13.49 (p95 ≈14.73).\n"
    "- trace.csv: 28 unique cmdb_id values, 50 unique trace_name values. Top trace entries (25 points each) are dominated by ts-travel2-service, "
    "e.g., row_count mean ≈2472.08 (p95 ≈2742.80); seat-service and train-service downstream durations show small mean latencies (tens of ms).\n"
    "- log.csv: 46 unique cmdb_id values, 2 log_name values (log.row_count, log.error_count). Many services show large log.row_count means (examples: ts-basic-service ≈2323.6, "
    "ts-travel2-service ≈2172.96, ts-route-service ≈959.04). log.error_count is 0 in the top entries.\n"
    "- error_logs.csv: no error log rows/messages were present (empty), so no message samples to report.\n\n"
    "Overall: data is present and uniform (many groups have 25 samples). Key hotspots to inspect further are high memory and workload/socket metrics on ts-travel2-service and "
    "high memory on ts-admin-basic-info-service. No error log content was captured in error_logs.csv."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(0     ts-admin-basic-info-service
1          ts-admin-order-service
2          ts-admin-route-service
3         ts-admin-travel-service
4           ts-admin-user-service
                 ...             
45          ts-route-plan-service
46               ts-route-service
47                ts-seat-service
48              ts-security-mongo
49            ts-security-service
Length: 50, dtype: object, 0           cpu
1        diskio
2    latency-50
3    latency-90
4           mem
5        socket
6      workload
7         error
dtype: object,                          cmdb_id    kpi_name  count             mean              p95
362           ts-voucher-service      socket     25         4.000000         4.000000
0    ts-admin-basic-info-service         cpu     25         0.388522         0.687403
1    ts-admin-basic-info-service      diskio     25     53172.273561     59816.719121
2    ts-admin-basic-info-service  latency-50     25         0.015427         0.016377
3    ts-admin-basic-info-service  latency-90     25         0.023119         0.023275
4    ts-admin-basic-info-service         mem     25 236405540.396912 237074582.186667
5    ts-admin-basic-info-service      socket     25         7.485930         8.180000
6    ts-admin-basic-info-service    workload     25         0.637074         0.727340
7         ts-admin-order-service         cpu     25         0.092535         0.106980
8         ts-admin-order-service      diskio     25     38225.471764     48776.638202
9         ts-admin-order-service         mem     25 193122820.096000 193188372.480000
323             ts-travel2-mongo      socket     25         6.000000         6.000000
324           ts-travel2-service         cpu     25         4.591521         5.101155
325           ts-travel2-service      diskio     25     51633.813243     63502.247641
326           ts-travel2-service  latency-50     25         0.019211         0.020423
327           ts-travel2-service  latency-90     25         0.471992         0.635158
328           ts-travel2-service         mem     25 279992040.178526 280196369.066667
329           ts-travel2-service      socket     25        19.686982        21.710000
330           ts-travel2-service    workload     25        13.490546        14.728730
315            ts-travel-service  latency-50     25         0.018290         0.019297, 0                            root
1     ts-admin-basic-info-service
2         ts-admin-travel-service
3            ts-assurance-service
4                 ts-auth-service
                 ...             
23          ts-ticketinfo-service
24               ts-train-service
25              ts-travel-service
26             ts-travel2-service
27                ts-user-service
Length: 28, dtype: object, 0     trace.to_ts-admin-basic-info-service.duration_...
1     trace.to_ts-admin-basic-info-service.duration_p95
2       trace.to_ts-admin-basic-info-service.error_rate
3        trace.to_ts-admin-basic-info-service.row_count
4        trace.to_ts-admin-travel-service.duration_mean
                            ...                        
45            trace.to_ts-preserve-service.duration_p95
46              trace.to_ts-preserve-service.error_rate
47               trace.to_ts-preserve-service.row_count
48             trace.to_ts-travel-service.duration_mean
49              trace.to_ts-travel-service.duration_p95
Length: 50, dtype: object,                 cmdb_id                                    trace_name  count        mean         p95
736  ts-travel2-service        trace.to_ts-seat-service.duration_mean     25    0.078120    0.122430
737  ts-travel2-service         trace.to_ts-seat-service.duration_p95     25    0.198779    0.447704
738  ts-travel2-service           trace.to_ts-seat-service.error_rate     25    0.000000    0.000000
739  ts-travel2-service            trace.to_ts-seat-service.row_count     25  149.200000  165.800000
740  ts-travel2-service  trace.to_ts-ticketinfo-service.duration_mean     25    0.018401    0.025609
741  ts-travel2-service   trace.to_ts-ticketinfo-service.duration_p95     25    0.046533    0.051667
742  ts-travel2-service     trace.to_ts-ticketinfo-service.error_rate     25    0.000000    0.000000
743  ts-travel2-service      trace.to_ts-ticketinfo-service.row_count     25  636.440000  709.000000
744  ts-travel2-service       trace.to_ts-train-service.duration_mean     25    0.028939    0.058643
745  ts-travel2-service        trace.to_ts-train-service.duration_p95     25    0.113178    0.213431
746  ts-travel2-service          trace.to_ts-train-service.error_rate     25    0.000000    0.000000
747  ts-travel2-service           trace.to_ts-train-service.row_count     25  272.080000  302.400000
748  ts-travel2-service     trace.to_ts-travel2-service.duration_mean     25    0.016374    0.025033
749  ts-travel2-service      trace.to_ts-travel2-service.duration_p95     25    0.062523    0.113978
750  ts-travel2-service        trace.to_ts-travel2-service.error_rate     25    0.000000    0.000000
751  ts-travel2-service         trace.to_ts-travel2-service.row_count     25 2472.080000 2742.800000
480    ts-price-service     trace.from_ts-basic-service.duration_mean     25    0.003648    0.003724
481    ts-price-service      trace.from_ts-basic-service.duration_p95     25    0.004371    0.004710
482    ts-price-service        trace.from_ts-basic-service.error_rate     25    0.000000    0.000000
483    ts-price-service         trace.from_ts-basic-service.row_count     25  127.160000  145.000000, 0     ts-admin-basic-info-service
1         ts-admin-travel-service
2              ts-assurance-mongo
3            ts-assurance-service
4                   ts-auth-mongo
                 ...             
41               ts-travel2-mongo
42             ts-travel2-service
43                ts-ui-dashboard
44                  ts-user-mongo
45                ts-user-service
Length: 46, dtype: object, 0    log.error_count
1      log.row_count
dtype: object,                     cmdb_id         log_name  count        mean         p95
3   ts-admin-travel-service    log.row_count     25   27.480000   30.800000
2   ts-admin-travel-service  log.error_count     25    0.000000    0.000000
11          ts-auth-service    log.row_count     25  315.120000  343.200000
10          ts-auth-service  log.error_count     25    0.000000    0.000000
20       ts-consign-service  log.error_count     25    0.000000    0.000000
21       ts-consign-service    log.row_count     25   43.960000   63.000000
13         ts-basic-service    log.row_count     25 2323.600000 2628.800000
12         ts-basic-service  log.error_count     25    0.000000    0.000000
84       ts-travel2-service  log.error_count     25    0.000000    0.000000
85       ts-travel2-service    log.row_count     25 2172.960000 2401.400000
86          ts-ui-dashboard  log.error_count     25    0.000000    0.000000
87          ts-ui-dashboard    log.row_count     25  260.640000  287.400000
60         ts-price-service  log.error_count     25    0.000000    0.000000
61         ts-price-service    log.row_count     25  254.320000  290.000000
65         ts-route-service    log.row_count     25  959.040000 1064.600000
64         ts-route-service  log.error_count     25    0.000000    0.000000
66          ts-seat-service  log.error_count     25    0.000000    0.000000
67          ts-seat-service    log.row_count     25 1725.040000 1955.400000
80        ts-travel-service  log.error_count     25    0.000000    0.000000
81        ts-travel-service    log.row_count     25 1564.600000 1899.000000, Series([], dtype: object), Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [message, count]
Index: [])```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing DataFrames in kernel: metric_df, trace_df, log_df, error_logs_df
# Ensure timestamps are parsed (rule): parse with unit='s', utc=True
for df in (metric_df, trace_df, log_df, error_logs_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Candidate components
candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Incident window (UTC)
start_ts = pd.to_datetime("2024-01-24 20:13:03", utc=True)
end_ts   = pd.to_datetime("2024-01-24 20:43:03", utc=True)

# Helper to compute global p05/p95 per group for a dataframe with columns ['cmdb_id', name_col, 'value']
def global_thresholds(df, name_col):
    grp = df.groupby(['cmdb_id', name_col])['value']
    thresholds = grp.agg(global_count='count',
                         global_mean='mean',
                         global_p95=lambda x: x.quantile(0.95),
                         global_p05=lambda x: x.quantile(0.05)).reset_index()
    return thresholds

# A) metric.csv analysis
metric_candidates_df = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# List KPI names per candidate (compact)
metric_kpi_list_df = (
    metric_candidates_df
    .groupby('cmdb_id')['kpi_name']
    .unique()
    .reset_index()
)
# convert arrays to comma-joined strings (compact)
metric_kpi_list_df['kpi_names'] = metric_kpi_list_df['kpi_name'].apply(lambda arr: ','.join(arr[:50]) if len(arr)>0 else '')
metric_kpi_list_df = metric_kpi_list_df[['cmdb_id','kpi_names']]

# Compute global thresholds before filtering (rule 9)
if not metric_candidates_df.empty:
    metric_thresholds = global_thresholds(metric_candidates_df, 'kpi_name')
else:
    metric_thresholds = pd.DataFrame(columns=['cmdb_id','kpi_name','global_count','global_mean','global_p95','global_p05'])

# Filter to incident window
metric_window = metric_df[
    (metric_df['cmdb_id'].isin(candidates)) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# If window empty, produce empty anomalies table
if metric_window.empty or metric_thresholds.empty:
    metric_anomalies_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','global_p05','window_count','anomaly_count',
        'earliest_anomaly_timestamp','max_value','max_value_timestamp','breach_ratio',
        'min_value','min_value_timestamp','drop_ratio'
    ])
else:
    # Merge thresholds into window rows
    metric_window = metric_window.merge(metric_thresholds[['cmdb_id','kpi_name','global_p95','global_p05']], on=['cmdb_id','kpi_name'], how='left')
    # Define KPI types
    def is_resource_latency(k):
        if pd.isna(k): 
            return False
        k2 = k.lower()
        return any(tok in k2 for tok in ['cpu','mem','diskio','socket','latency','error'])
    def is_workload(k):
        if pd.isna(k):
            return False
        k2 = k.lower()
        return any(tok in k2 for tok in ['workload','row_count'])
    metric_window['is_reslat'] = metric_window['kpi_name'].apply(is_resource_latency)
    metric_window['is_workload'] = metric_window['kpi_name'].apply(is_workload)

    # Mark anomalies per row
    # For resource/latency: value > global_p95
    metric_window['anomaly_high'] = False
    metric_window['anomaly_low'] = False
    mask_p95 = metric_window['global_p95'].notna()
    metric_window.loc[mask_p95 & metric_window['is_reslat'] & (metric_window['value'] > metric_window['global_p95']), 'anomaly_high'] = True
    mask_p05 = metric_window['global_p05'].notna()
    metric_window.loc[mask_p05 & metric_window['is_workload'] & (metric_window['value'] <= metric_window['global_p05']), 'anomaly_low'] = True

    # Aggregate per group
    agg_rows = []
    grouped = metric_window.groupby(['cmdb_id','kpi_name','global_p95','global_p05'])
    for (cmdb_id,kpi_name,global_p95,global_p05), grp in grouped:
        window_count = len(grp)
        anomaly_high_rows = grp[grp['anomaly_high']]
        anomaly_low_rows  = grp[grp['anomaly_low']]
        anomaly_count = int(len(anomaly_high_rows) + len(anomaly_low_rows))
        if anomaly_count == 0:
            continue
        # earliest anomaly timestamp (any type)
        if not anomaly_high_rows.empty:
            earliest_anom_ts_high = anomaly_high_rows['timestamp'].min()
        else:
            earliest_anom_ts_high = pd.NaT
        if not anomaly_low_rows.empty:
            earliest_anom_ts_low = anomaly_low_rows['timestamp'].min()
        else:
            earliest_anom_ts_low = pd.NaT
        # choose earliest overall
        earliest_anom_ts_candidates = [t for t in (earliest_anom_ts_high, earliest_anom_ts_low) if (pd.notna(t))]
        earliest_anomaly_timestamp = min(earliest_anom_ts_candidates) if earliest_anom_ts_candidates else pd.NaT

        # For high anomalies compute max_value and timestamp
        if not anomaly_high_rows.empty:
            idx_max = anomaly_high_rows['value'].idxmax()
            max_value = float(anomaly_high_rows.loc[idx_max,'value'])
            max_value_timestamp = anomaly_high_rows.loc[idx_max,'timestamp']
            breach_ratio = (max_value - global_p95)/global_p95 if (global_p95 and global_p95!=0) else np.nan
        else:
            max_value = np.nan
            max_value_timestamp = pd.NaT
            breach_ratio = np.nan

        # For low anomalies compute min_value and timestamp
        if not anomaly_low_rows.empty:
            idx_min = anomaly_low_rows['value'].idxmin()
            min_value = float(anomaly_low_rows.loc[idx_min,'value'])
            min_value_timestamp = anomaly_low_rows.loc[idx_min,'timestamp']
            drop_ratio = (global_p05 - min_value)/global_p05 if (global_p05 and global_p05!=0) else np.nan
        else:
            min_value = np.nan
            min_value_timestamp = pd.NaT
            drop_ratio = np.nan

        agg_rows.append({
            'cmdb_id': cmdb_id,
            'kpi_name': kpi_name,
            'global_p95': float(global_p95) if pd.notna(global_p95) else np.nan,
            'global_p05': float(global_p05) if pd.notna(global_p05) else np.nan,
            'window_count': int(window_count),
            'anomaly_count': int(anomaly_count),
            'earliest_anomaly_timestamp': earliest_anomaly_timestamp,
            'max_value': max_value,
            'max_value_timestamp': max_value_timestamp,
            'breach_ratio': breach_ratio,
            'min_value': min_value,
            'min_value_timestamp': min_value_timestamp,
            'drop_ratio': drop_ratio
        })

    metric_anomalies_df = pd.DataFrame(agg_rows)
    if metric_anomalies_df.empty:
        # ensure consistent columns
        metric_anomalies_df = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','global_p95','global_p05','window_count','anomaly_count',
            'earliest_anomaly_timestamp','max_value','max_value_timestamp','breach_ratio',
            'min_value','min_value_timestamp','drop_ratio'
        ])
    else:
        # severity score for sorting
        metric_anomalies_df['severity'] = metric_anomalies_df[['breach_ratio','drop_ratio']].max(axis=1, skipna=True)
        metric_anomalies_df = metric_anomalies_df.sort_values('severity', ascending=False).head(20)
        metric_anomalies_df = metric_anomalies_df.drop(columns=['severity'])

# B) trace.csv analysis
trace_candidates_df = trace_df[trace_df['cmdb_id'].isin(candidates)].copy()
trace_name_list_df = (
    trace_candidates_df
    .groupby('cmdb_id')['trace_name']
    .unique()
    .reset_index()
)
trace_name_list_df['trace_names'] = trace_name_list_df['trace_name'].apply(lambda arr: ','.join(arr[:50]) if len(arr)>0 else '')
trace_name_list_df = trace_name_list_df[['cmdb_id','trace_names']]

if not trace_candidates_df.empty:
    trace_thresholds = global_thresholds(trace_candidates_df, 'trace_name')
else:
    trace_thresholds = pd.DataFrame(columns=['cmdb_id','trace_name','global_count','global_mean','global_p95','global_p05'])

trace_window = trace_df[
    (trace_df['cmdb_id'].isin(candidates)) &
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

if trace_window.empty or trace_thresholds.empty:
    trace_anomalies_df = pd.DataFrame(columns=[
        'cmdb_id','trace_name','global_p95','global_p05','window_count','anomaly_count',
        'earliest_anomaly_timestamp','max_or_min_value','value_timestamp','severity_ratio'
    ])
else:
    trace_window = trace_window.merge(trace_thresholds[['cmdb_id','trace_name','global_p95','global_p05']], on=['cmdb_id','trace_name'], how='left')

    def trace_is_duration_like(name):
        if pd.isna(name): return False
        n = name.lower()
        return any(tok in n for tok in ['duration','duration_mean','duration_p95','error_rate'])
    def trace_is_rowcount(name):
        if pd.isna(name): return False
        return 'row_count' in name.lower()

    trace_window['is_dur'] = trace_window['trace_name'].apply(trace_is_duration_like)
    trace_window['is_rowcount'] = trace_window['trace_name'].apply(trace_is_rowcount)

    trace_window['anomaly_high'] = False
    trace_window['anomaly_low'] = False
    mask_t_p95 = trace_window['global_p95'].notna()
    trace_window.loc[mask_t_p95 & trace_window['is_dur'] & (trace_window['value'] > trace_window['global_p95']), 'anomaly_high'] = True
    mask_t_p05 = trace_window['global_p05'].notna()
    trace_window.loc[mask_t_p05 & trace_window['is_rowcount'] & (trace_window['value'] <= trace_window['global_p05']), 'anomaly_low'] = True

    agg_rows = []
    grouped = trace_window.groupby(['cmdb_id','trace_name','global_p95','global_p05'])
    for (cmdb_id, trace_name, global_p95, global_p05), grp in grouped:
        window_count = len(grp)
        anomaly_high_rows = grp[grp['anomaly_high']]
        anomaly_low_rows  = grp[grp['anomaly_low']]
        anomaly_count = int(len(anomaly_high_rows) + len(anomaly_low_rows))
        if anomaly_count == 0:
            continue
        earliest_ts_candidates = []
        if not anomaly_high_rows.empty:
            earliest_ts_candidates.append(anomaly_high_rows['timestamp'].min())
        if not anomaly_low_rows.empty:
            earliest_ts_candidates.append(anomaly_low_rows['timestamp'].min())
        earliest_anomaly_timestamp = min(earliest_ts_candidates) if earliest_ts_candidates else pd.NaT

        if not anomaly_high_rows.empty:
            idx_max = anomaly_high_rows['value'].idxmax()
            max_or_min_value = float(anomaly_high_rows.loc[idx_max,'value'])
            value_timestamp = anomaly_high_rows.loc[idx_max,'timestamp']
            severity_ratio = (max_or_min_value - global_p95)/global_p95 if (global_p95 and global_p95!=0) else np.nan
        else:
            idx_min = anomaly_low_rows['value'].idxmin()
            max_or_min_value = float(anomaly_low_rows.loc[idx_min,'value'])
            value_timestamp = anomaly_low_rows.loc[idx_min,'timestamp']
            severity_ratio = (global_p05 - max_or_min_value)/global_p05 if (global_p05 and global_p05!=0) else np.nan

        agg_rows.append({
            'cmdb_id': cmdb_id,
            'trace_name': trace_name,
            'global_p95': float(global_p95) if pd.notna(global_p95) else np.nan,
            'global_p05': float(global_p05) if pd.notna(global_p05) else np.nan,
            'window_count': int(window_count),
            'anomaly_count': int(anomaly_count),
            'earliest_anomaly_timestamp': earliest_anomaly_timestamp,
            'max_or_min_value': max_or_min_value,
            'value_timestamp': value_timestamp,
            'severity_ratio': severity_ratio
        })

    trace_anomalies_df = pd.DataFrame(agg_rows)
    if trace_anomalies_df.empty:
        trace_anomalies_df = pd.DataFrame(columns=[
            'cmdb_id','trace_name','global_p95','global_p05','window_count','anomaly_count',
            'earliest_anomaly_timestamp','max_or_min_value','value_timestamp','severity_ratio'
        ])
    else:
        trace_anomalies_df = trace_anomalies_df.sort_values('severity_ratio', ascending=False).head(20)

# C) log.csv analysis
log_candidates_df = log_df[log_df['cmdb_id'].isin(candidates)].copy()
log_name_list_df = (
    log_candidates_df
    .groupby('cmdb_id')['log_name']
    .unique()
    .reset_index()
)
log_name_list_df['log_names'] = log_name_list_df['log_name'].apply(lambda arr: ','.join(arr[:50]) if len(arr)>0 else '')
log_name_list_df = log_name_list_df[['cmdb_id','log_names']]

if not log_candidates_df.empty:
    log_thresholds = global_thresholds(log_candidates_df, 'log_name')
else:
    log_thresholds = pd.DataFrame(columns=['cmdb_id','log_name','global_count','global_mean','global_p95','global_p05'])

log_window = log_df[
    (log_df['cmdb_id'].isin(candidates)) &
    (log_df['timestamp'] >= start_ts) &
    (log_df['timestamp'] <= end_ts)
].copy()

if log_window.empty or log_thresholds.empty:
    log_anomalies_df = pd.DataFrame(columns=[
        'cmdb_id','log_name','global_p95','global_p05','window_count','anomaly_count',
        'earliest_anomaly_timestamp','max_or_min_value','value_timestamp','severity_ratio'
    ])
else:
    log_window = log_window.merge(log_thresholds[['cmdb_id','log_name','global_p95','global_p05']], on=['cmdb_id','log_name'], how='left')

    log_window['anomaly_high'] = False
    log_window['anomaly_low'] = False
    mask_lp95 = log_window['global_p95'].notna()
    log_window.loc[mask_lp95 & (log_window['log_name']=='log.error_count') & (log_window['value'] > log_window['global_p95']), 'anomaly_high'] = True
    mask_lp05 = log_window['global_p05'].notna()
    log_window.loc[mask_lp05 & (log_window['log_name']=='log.row_count') & (log_window['value'] <= log_window['global_p05']), 'anomaly_low'] = True

    agg_rows = []
    grouped = log_window.groupby(['cmdb_id','log_name','global_p95','global_p05'])
    for (cmdb_id, log_name, global_p95, global_p05), grp in grouped:
        window_count = len(grp)
        anomaly_high_rows = grp[grp['anomaly_high']]
        anomaly_low_rows  = grp[grp['anomaly_low']]
        anomaly_count = int(len(anomaly_high_rows) + len(anomaly_low_rows))
        if anomaly_count == 0:
            continue
        earliest_ts_candidates = []
        if not anomaly_high_rows.empty:
            earliest_ts_candidates.append(anomaly_high_rows['timestamp'].min())
        if not anomaly_low_rows.empty:
            earliest_ts_candidates.append(anomaly_low_rows['timestamp'].min())
        earliest_anomaly_timestamp = min(earliest_ts_candidates) if earliest_ts_candidates else pd.NaT

        if not anomaly_high_rows.empty:
            idx_max = anomaly_high_rows['value'].idxmax()
            max_or_min_value = float(anomaly_high_rows.loc[idx_max,'value'])
            value_timestamp = anomaly_high_rows.loc[idx_max,'timestamp']
            severity_ratio = (max_or_min_value - global_p95)/global_p95 if (global_p95 and global_p95!=0) else np.nan
        else:
            idx_min = anomaly_low_rows['value'].idxmin()
            max_or_min_value = float(anomaly_low_rows.loc[idx_min,'value'])
            value_timestamp = anomaly_low_rows.loc[idx_min,'timestamp']
            severity_ratio = (global_p05 - max_or_min_value)/global_p05 if (global_p05 and global_p05!=0) else np.nan

        agg_rows.append({
            'cmdb_id': cmdb_id,
            'log_name': log_name,
            'global_p95': float(global_p95) if pd.notna(global_p95) else np.nan,
            'global_p05': float(global_p05) if pd.notna(global_p05) else np.nan,
            'window_count': int(window_count),
            'anomaly_count': int(anomaly_count),
            'earliest_anomaly_timestamp': earliest_anomaly_timestamp,
            'max_or_min_value': max_or_min_value,
            'value_timestamp': value_timestamp,
            'severity_ratio': severity_ratio
        })

    log_anomalies_df = pd.DataFrame(agg_rows)
    if log_anomalies_df.empty:
        log_anomalies_df = pd.DataFrame(columns=[
            'cmdb_id','log_name','global_p95','global_p05','window_count','anomaly_count',
            'earliest_anomaly_timestamp','max_or_min_value','value_timestamp','severity_ratio'
        ])
    else:
        log_anomalies_df = log_anomalies_df.sort_values('severity_ratio', ascending=False).head(20)

# Final outputs (compact). If empty, they will be empty DataFrames as required.
metric_kpi_list_df, metric_anomalies_df, trace_name_list_df, trace_anomalies_df, log_name_list_df, log_anomalies_df

```
Out[3]:
```
Summary of anomalies for the five candidate components during 2024-01-24 20:13:03 → 20:43:03 UTC

Key overall findings
- There is a consistent pattern of large drops in traffic/row_count across many services inside the incident window (trace row_count and log.row_count anomalies with very high severity ratios ~0.9). This affects ts-auth-service, ts-order-service, ts-route-service, ts-train-service, and ts-travel-service.
- In addition to traffic drops, several services show resource/latency breaches (> global P95) in the same window (notably ts-order-service, ts-train-service, ts-travel-service, ts-auth-service).

Per-component highlights

1) ts-auth-service
- Metrics: CPU and disk I/O exceeded their global P95 during the window (cpu breach_ratio ≈ 0.062; diskio breach_ratio ≈ 0.132). Earliest metric anomalies around 2024-01-24 20:16:00–20:22:00; maximum CPU observed ~16.58 at 20:36 (window max timestamp) and diskio max ~64,368 at 20:40.
- Traces: Row_count drops for traces (trace.from_root.row_count and to/from auth row_count) with severity ≈ 0.933 (min row_count ≈ 5 or 20 at 20:40).
- Logs: log.row_count dropped (severity ≈ 0.934), earliest anomaly ~2024-01-24 20:32:00; min row_count ≈ 20 at 2024-01-24 20:40:00.
- Overall: traffic drop coincident with moderate CPU/disk spikes.

2) ts-order-service
- Metrics: latency-90 spiked strongly (breach_ratio ≈ 1.763 — ~176% above p95), diskio and cpu also breached p95 (diskio breach_ratio ≈ 0.135, cpu ≈ 0.098). Earliest metric anomalies around 20:19–20:27; some maxs at ~20:26–20:40.
- Workload: workload dropped below global P05 (min 3.5 at 20:40, drop_ratio ≈ 0.20).
- Traces: trace.from_ts-inside-payment-service.duration_p95 and duration_mean show large spikes (severity up to ≈3.31 and ≈1.22) with earliest anomaly ~20:19.
- Trace & Log row_count: major row_count drops (trace and log row_count) with severity ≈0.92; min row_count ≈22 at 20:40.
- Overall: pronounced latency spike in ordering flows plus significant traffic drop.

3) ts-route-service
- Metrics: small latency and CPU/diskio breaches present but less severe (latency-90 breach_ratio ≈ 0.052; latency-50 breach_ratio ≈ 0.041).
- Traces: large row_count drops for multiple trace directions (severity ≈0.924–0.925), earliest anomalies ~20:25–20:37; min row_count examples: 40 and 136 at 20:40.
- Logs: log.row_count dropped (severity ≈0.924), earliest anomaly ~20:37; min row_count ≈68 at 20:40.
- Overall: primarily a traffic/row_count degradation with minor latency increases.

4) ts-train-service
- Metrics: diskio and cpu breached p95 substantially (diskio breach_ratio ≈ 1.621; cpu breach_ratio ≈ 0.247). Earliest anomalies ~20:16:00 with high diskio observed (max ~172,106 at 20:16).
- Workload: small drop below P05 (drop_ratio ≈ 0.049).
- Traces: trace row_count drops (severity ≈0.911) with min row_count = 194 at 20:40; earliest anomaly ~20:37.
- Overall: strong disk I/O spike early in the window plus row_count drop later in the window.

5) ts-travel-service
- Metrics: CPU and diskio exceeded p95 (cpu breach_ratio ≈ 0.354; diskio breach_ratio ≈ 0.233); socket and latency-90 also slightly above p95 in window. Earliest metric anomalies ~20:17–20:27; max cpu ~5.20 at 20:27, diskio max ~72,559 at 20:40.
- Workload: workload dropped below P05 (min ≈6.13 at 20:40, drop_ratio ≈ 0.056).
- Traces: duration_p95 spikes for some downstream traces (e.g., food-service duration_p95 with severity ≈1.725) and row_count drops for travel traces (severity ≈0.905), earliest trace anomalies ~20:25–20:26.
- Logs: log.row_count dropped (severity ≈0.907), min row_count ≈109 at 20:40.
- Overall: combined CPU/diskio spikes and downstream latency spikes plus traffic drop.

Additional note
- error_logs.csv contained no messages, so no log-error samples were available.

Concise conclusion
- The dominant, consistent signal across all five components during the incident window is a severe drop in traffic/row_count (trace row_count and log.row_count) with high severity ratios (~0.9), indicating large reductions in served requests across many services.
- Coincident with that traffic drop, several services show resource/latency breaches (most notable: ts-order-service latency-90 spike; ts-train-service diskio spike; ts-travel-service CPU/diskio and some trace duration spikes).
- Recommended next steps (if you want actions): investigate upstream traffic source or routing changes that could cause the widespread row_count drop, and inspect ts-order-service and ts-train-service resource/latency spikes for local issues or contention during the window.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id                                          kpi_names
0    ts-auth-service  cpu,diskio,latency-50,latency-90,mem,socket,wo...
1   ts-order-service  cpu,diskio,error,latency-50,latency-90,mem,soc...
2   ts-route-service  cpu,diskio,latency-50,latency-90,mem,socket,wo...
3   ts-train-service  cpu,diskio,latency-50,latency-90,mem,socket,wo...
4  ts-travel-service  cpu,diskio,error,latency-50,latency-90,mem,soc...,               cmdb_id    kpi_name   global_p95   global_p05  window_count  anomaly_count earliest_anomaly_timestamp     max_value       max_value_timestamp  breach_ratio  min_value       min_value_timestamp  drop_ratio
10   ts-order-service  latency-90     0.110697     0.021878            25              2  2024-01-24 20:25:00+00:00      0.305866 2024-01-24 20:26:00+00:00      1.763105        NaN                       NaT         NaN
22   ts-train-service      diskio 65659.300790 42618.618404            25              2  2024-01-24 20:16:00+00:00 172106.736341 2024-01-24 20:16:00+00:00      1.621209        NaN                       NaT         NaN
28  ts-travel-service         cpu     3.840675     2.530666            25              2  2024-01-24 20:27:00+00:00      5.199324 2024-01-24 20:27:00+00:00      0.353752        NaN                       NaT         NaN
21   ts-train-service         cpu     3.231628     2.715023            25              2  2024-01-24 20:16:00+00:00      4.030889 2024-01-24 20:16:00+00:00      0.247325        NaN                       NaT         NaN
29  ts-travel-service      diskio 58850.107261 42903.663223            25              2  2024-01-24 20:17:00+00:00  72559.787422 2024-01-24 20:40:00+00:00      0.232959        NaN                       NaT         NaN
13   ts-order-service    workload     7.154910     4.375120            25              2  2024-01-24 20:25:00+00:00           NaN                       NaT           NaN   3.500000 2024-01-24 20:40:00+00:00    0.200022
33  ts-travel-service      socket    18.383333    14.390000            25              2  2024-01-24 20:32:00+00:00     21.150000 2024-01-24 20:32:00+00:00      0.150499        NaN                       NaT         NaN
8    ts-order-service      diskio 60331.456355 42639.667694            25              2  2024-01-24 20:33:00+00:00  68476.375567 2024-01-24 20:40:00+00:00      0.135003        NaN                       NaT         NaN
1     ts-auth-service      diskio 56886.251451 39136.188435            25              2  2024-01-24 20:30:00+00:00  64368.357986 2024-01-24 20:40:00+00:00      0.131528        NaN                       NaT         NaN
9    ts-order-service  latency-50     0.010063     0.007364            25              2  2024-01-24 20:27:00+00:00      0.011329 2024-01-24 20:35:00+00:00      0.125761        NaN                       NaT         NaN
31  ts-travel-service  latency-90     0.556898     0.089956            25              2  2024-01-24 20:29:00+00:00      0.621041 2024-01-24 20:35:00+00:00      0.115180        NaN                       NaT         NaN
12   ts-order-service      socket    12.343333    10.023333            25              2  2024-01-24 20:18:00+00:00     13.600000 2024-01-24 20:18:00+00:00      0.101809        NaN                       NaT         NaN
7    ts-order-service         cpu     2.628109     1.427757            25              2  2024-01-24 20:20:00+00:00      2.884699 2024-01-24 20:20:00+00:00      0.097633        NaN                       NaT         NaN
0     ts-auth-service         cpu    15.616384    12.396155            25              2  2024-01-24 20:16:00+00:00     16.580082 2024-01-24 20:36:00+00:00      0.061711        NaN                       NaT         NaN
34  ts-travel-service    workload    10.206637     6.496207            25              2  2024-01-24 20:25:00+00:00           NaN                       NaT           NaN   6.133500 2024-01-24 20:40:00+00:00    0.055834
17   ts-route-service  latency-90     0.007862     0.007294            25              2  2024-01-24 20:17:00+00:00      0.008269 2024-01-24 20:17:00+00:00      0.051707        NaN                       NaT         NaN
27   ts-train-service    workload    22.103417    19.071360            25              2  2024-01-24 20:26:00+00:00           NaN                       NaT           NaN  18.129850 2024-01-24 20:34:00+00:00    0.049368
5     ts-auth-service      socket    16.569825    13.946667            25              2  2024-01-24 20:22:00+00:00     17.366667 2024-01-24 20:35:00+00:00      0.048090        NaN                       NaT         NaN
16   ts-route-service  latency-50     0.003543     0.003421            25              2  2024-01-24 20:17:00+00:00      0.003689 2024-01-24 20:17:00+00:00      0.041435        NaN                       NaT         NaN
3     ts-auth-service  latency-90     0.849446     0.462770            25              2  2024-01-24 20:35:00+00:00      0.880568 2024-01-24 20:35:00+00:00      0.036637        NaN                       NaT         NaN,              cmdb_id                                        trace_names
0    ts-auth-service  trace.from_root.duration_mean,trace.from_root....
1   ts-order-service  trace.from_root.duration_mean,trace.from_root....
2   ts-route-service  trace.from_ts-basic-service.duration_mean,trac...
3   ts-train-service  trace.from_ts-basic-service.duration_mean,trac...
4  ts-travel-service  trace.from_root.duration_mean,trace.from_root....,                cmdb_id                                         trace_name  global_p95  global_p05  window_count  anomaly_count earliest_anomaly_timestamp  max_or_min_value           value_timestamp  severity_ratio
13    ts-order-service  trace.from_ts-inside-payment-service.duration_p95    0.010223    0.003509            23              2  2024-01-24 20:19:00+00:00          0.044096 2024-01-24 20:19:00+00:00        3.313534
73   ts-travel-service            trace.from_ts-food-service.duration_p95    0.015931    0.010719            25              2  2024-01-24 20:26:00+00:00          0.043408 2024-01-24 20:26:00+00:00        1.724682
12    ts-order-service  trace.from_ts-inside-payment-service.duration_...    0.005469    0.003504            23              2  2024-01-24 20:19:00+00:00          0.012118 2024-01-24 20:19:00+00:00        1.215663
53    ts-train-service              trace.from_ts-basic-service.row_count  145.000000  114.400000            25              2  2024-01-24 20:25:00+00:00          5.000000 2024-01-24 20:40:00+00:00        0.956294
38    ts-route-service              trace.from_ts-basic-service.row_count  145.000000  114.400000            25              2  2024-01-24 20:25:00+00:00          6.000000 2024-01-24 20:40:00+00:00        0.947552
5      ts-auth-service               trace.from_ts-auth-service.row_count  343.200000  301.600000            25              2  2024-01-24 20:32:00+00:00         20.000000 2024-01-24 20:40:00+00:00        0.933687
2      ts-auth-service                          trace.from_root.row_count   85.800000   75.400000            25              2  2024-01-24 20:32:00+00:00          5.000000 2024-01-24 20:40:00+00:00        0.933687
8      ts-auth-service                 trace.to_ts-auth-service.row_count  343.200000  301.600000            25              2  2024-01-24 20:32:00+00:00         20.000000 2024-01-24 20:40:00+00:00        0.933687
95   ts-travel-service           trace.to_ts-ticketinfo-service.row_count  356.800000  188.600000            25              2  2024-01-24 20:25:00+00:00         14.000000 2024-01-24 20:40:00+00:00        0.925769
47    ts-route-service            trace.from_ts-travel2-service.row_count  614.000000  535.800000            25              2  2024-01-24 20:34:00+00:00         40.000000 2024-01-24 20:40:00+00:00        0.925345
41    ts-route-service              trace.from_ts-route-service.row_count 2129.200000 1802.000000            25              2  2024-01-24 20:37:00+00:00        136.000000 2024-01-24 20:40:00+00:00        0.924528
50    ts-route-service                trace.to_ts-route-service.row_count 2129.200000 1802.000000            25              2  2024-01-24 20:37:00+00:00        136.000000 2024-01-24 20:40:00+00:00        0.924528
32    ts-order-service                trace.to_ts-order-service.row_count  530.600000  289.800000            25              2  2024-01-24 20:25:00+00:00         22.000000 2024-01-24 20:40:00+00:00        0.924086
17    ts-order-service              trace.from_ts-order-service.row_count  530.600000  289.800000            25              2  2024-01-24 20:25:00+00:00         22.000000 2024-01-24 20:40:00+00:00        0.924086
56    ts-train-service              trace.from_ts-train-service.row_count 2620.800000 2180.000000            25              2  2024-01-24 20:37:00+00:00        194.000000 2024-01-24 20:40:00+00:00        0.911009
65    ts-train-service                trace.to_ts-train-service.row_count 2620.800000 2180.000000            25              2  2024-01-24 20:37:00+00:00        194.000000 2024-01-24 20:40:00+00:00        0.911009
62    ts-train-service            trace.from_ts-travel2-service.row_count  302.400000  257.200000            25              2  2024-01-24 20:36:00+00:00         24.000000 2024-01-24 20:40:00+00:00        0.906687
83   ts-travel-service             trace.from_ts-travel-service.row_count 1689.400000 1005.200000            25              2  2024-01-24 20:25:00+00:00         95.000000 2024-01-24 20:40:00+00:00        0.905491
101  ts-travel-service               trace.to_ts-travel-service.row_count 1689.400000 1005.200000            25              2  2024-01-24 20:25:00+00:00         95.000000 2024-01-24 20:40:00+00:00        0.905491
29    ts-order-service             trace.from_ts-travel-service.row_count   59.200000   31.400000            25              2  2024-01-24 20:25:00+00:00          3.000000 2024-01-24 20:40:00+00:00        0.904459,              cmdb_id                      log_names
0    ts-auth-service  log.error_count,log.row_count
1   ts-order-service  log.error_count,log.row_count
2   ts-route-service  log.error_count,log.row_count
3  ts-travel-service  log.error_count,log.row_count,              cmdb_id       log_name  global_p95  global_p05  window_count  anomaly_count earliest_anomaly_timestamp  max_or_min_value           value_timestamp  severity_ratio
0    ts-auth-service  log.row_count  343.200000  301.600000            25              2  2024-01-24 20:32:00+00:00         20.000000 2024-01-24 20:40:00+00:00        0.933687
2   ts-route-service  log.row_count 1064.600000  901.000000            25              2  2024-01-24 20:37:00+00:00         68.000000 2024-01-24 20:40:00+00:00        0.924528
1   ts-order-service  log.row_count  432.000000  237.200000            25              2  2024-01-24 20:25:00+00:00         22.000000 2024-01-24 20:40:00+00:00        0.907251
3  ts-travel-service  log.row_count 1899.000000 1167.400000            25              2  2024-01-24 20:25:00+00:00        109.000000 2024-01-24 20:40:00+00:00        0.906630)```
```

In [ ]:
# In[4]:

import re
import numpy as np
import pandas as pd

# Reuse trace_df and trace_thresholds computed earlier in the session.
# Ensure timestamps parsed (they should be), and thresholds exist.
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

candidates = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]
start_ts = pd.to_datetime("2024-01-24 20:13:03", utc=True)
end_ts   = pd.to_datetime("2024-01-24 20:43:03", utc=True)

# Prepare candidate windowed trace rows
trace_window = trace_df[
    (trace_df['cmdb_id'].isin(candidates)) &
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# Merge global thresholds (must have been computed earlier as trace_thresholds)
# trace_thresholds columns: ['cmdb_id','trace_name','global_count','global_mean','global_p95','global_p05']
trace_window = trace_window.merge(
    trace_thresholds[['cmdb_id','trace_name','global_p95','global_p05']],
    on=['cmdb_id','trace_name'],
    how='left'
)

# Define anomaly types
def is_rowcount(name):
    return isinstance(name, str) and 'row_count' in name.lower()
def is_duration_like(name):
    if not isinstance(name, str): 
        return False
    n = name.lower()
    return any(tok in n for tok in ['duration','duration_mean','duration_p95','error_rate'])

trace_window['is_rowcount'] = trace_window['trace_name'].apply(is_rowcount)
trace_window['is_duration'] = trace_window['trace_name'].apply(is_duration_like)

# Determine anomalies using global thresholds (global p95/p05 computed from full series)
# Row_count anomaly: value <= global_p05
trace_window['anom_row_drop'] = trace_window['is_rowcount'] & trace_window['global_p05'].notna() & (trace_window['value'] <= trace_window['global_p05'])
# Duration/error anomaly: value > global_p95
trace_window['anom_dur_spike'] = trace_window['is_duration'] & trace_window['global_p95'].notna() & (trace_window['value'] > trace_window['global_p95'])

# Keep only anomalous rows
anomalous_rows = trace_window[ trace_window['anom_row_drop'] | trace_window['anom_dur_spike'] ].copy()
if anomalous_rows.empty:
    dest_agg_df = pd.DataFrame(columns=[
        'destination_service','anomaly_count','earliest_anomaly_timestamp',
        'example_min_row_count','example_max_duration','mean_severity_ratio'
    ])
    top_dest_examples = pd.DataFrame(columns=['trace_name','timestamp','value'])
else:
    # Parse destination service:
    # - if 'to_ts-<svc>' in trace_name -> destination = <svc>
    # - elif 'from_ts-<svc>' or 'from_root' -> destination = the cmdb_id (current service receiving)
    def parse_destination(row):
        name = row['trace_name'] if isinstance(row['trace_name'], str) else ''
        # Search to_ts- pattern
        m = re.search(r'to_ts-([a-z0-9\-]+)', name)
        if m:
            return m.group(1)
        m2 = re.search(r'from_ts-([a-z0-9\-]+)', name)
        if m2:
            # this indicates origin; destination is current cmdb_id
            return row['cmdb_id']
        if 'from_root' in name:
            return row['cmdb_id']
        # fallback: if trace_name contains 'to_' generic token, try to capture token after 'to_'
        m3 = re.search(r'to[_\-]([a-z0-9\-]+)', name)
        if m3:
            return m3.group(1)
        return row['cmdb_id']  # conservative fallback

    anomalous_rows['destination_service'] = anomalous_rows.apply(parse_destination, axis=1)

    # Compute severity per anomalous row
    def compute_severity(row):
        if row['anom_dur_spike']:
            gp95 = row['global_p95']
            if pd.notna(gp95) and gp95 != 0:
                return (row['value'] - gp95) / gp95
            return np.nan
        if row['anom_row_drop']:
            gp05 = row['global_p05']
            if pd.notna(gp05) and gp05 != 0:
                return (gp05 - row['value']) / gp05
            return np.nan
        return np.nan

    anomalous_rows['severity'] = anomalous_rows.apply(compute_severity, axis=1)

    # For grouping by destination_service compute required aggregates
    grouped = anomalous_rows.groupby('destination_service')

    agg_list = []
    for dest, grp in grouped:
        anomaly_count = len(grp)
        earliest_anomaly_timestamp = grp['timestamp'].min()
        # example_min_row_count: min value among row_count anomalies for this dest (if any)
        rowcount_grp = grp[ grp['anom_row_drop'] ]
        if not rowcount_grp.empty:
            example_min_row_count = float(rowcount_grp['value'].min())
        else:
            example_min_row_count = np.nan
        # example_max_duration: max value among duration anomalies for this dest (if any)
        dur_grp = grp[ grp['anom_dur_spike'] ]
        if not dur_grp.empty:
            example_max_duration = float(dur_grp['value'].max())
        else:
            example_max_duration = np.nan
        # mean severity ratio across all anomalies for this dest
        mean_severity_ratio = float(grp['severity'].dropna().mean()) if grp['severity'].notna().any() else np.nan

        agg_list.append({
            'destination_service': dest,
            'anomaly_count': int(anomaly_count),
            'earliest_anomaly_timestamp': earliest_anomaly_timestamp,
            'example_min_row_count': example_min_row_count,
            'example_max_duration': example_max_duration,
            'mean_severity_ratio': mean_severity_ratio
        })

    dest_agg_df = pd.DataFrame(agg_list)
    dest_agg_df = dest_agg_df.sort_values('anomaly_count', ascending=False).head(20)

    # Prepare up to 10 trace examples for the top destination service
    top_dest = dest_agg_df['destination_service'].iloc[0] if not dest_agg_df.empty else None
    if top_dest:
        top_examples = anomalous_rows[anomalous_rows['destination_service'] == top_dest].sort_values('timestamp').head(10)
        top_dest_examples = top_examples[['trace_name','timestamp','value']].copy()
    else:
        top_dest_examples = pd.DataFrame(columns=['trace_name','timestamp','value'])

# Return compact outputs
dest_agg_df.head(20), top_dest_examples.head(10)

```
Out[4]:
```
Summary of anomalous trace activity (2024-01-24 20:13:03 → 20:43:03 UTC)

Top affected downstream services (grouped by destination_service)
- ts-order-service: 48 anomalous trace rows (earliest anomaly 2024-01-24 20:16:00 UTC). Example signals: row_count dropped to 1 (example_min_row_count = 1.0) and duration spikes up to ~0.856s (example_max_duration ≈ 0.8559). Mean severity ≈ 0.20 (≈20% average deviation from global thresholds).
- ts-travel-service: 38 anomalies (earliest 20:22:00). example_min_row_count = 1.0, example_max_duration ≈ 2.783, mean severity ≈ 0.19.
- ts-train-service: 24 anomalies (earliest 20:25:00). example_min_row_count = 5.0, example_max_duration ≈ 0.2338, mean severity ≈ 0.20.
- ts-route-service: 24 anomalies (earliest 20:16:00). example_min_row_count = 6.0, example_max_duration ≈ 0.0048, mean severity ≈ 0.167.
- Additional destinations with anomalies include auth/order/train/travel (variants without the ts- prefix), station-service, seat-service, ticketinfo-service, etc., with lower counts.

Nature of anomalies
- Many anomalies are row_count drops (value <= global P05) indicating large decreases in traffic to those downstream services.
- There are also duration/error spikes (value > global P95) for some downstream calls (notably spikes seen targeting ts-travel-service and ts-order-service).
- The top destination (ts-order-service) shows both very low row_counts and notable duration spikes, suggesting both traffic reduction and higher latency for some call paths.

Example anomalous trace rows for the top destination (ts-order-service) — trace_name, timestamp (UTC), value
- trace.from_ts-inside-payment-service.row_count — 2024-01-24 20:16:00 — 1.0
- trace.from_ts-security-service.duration_mean — 2024-01-24 20:16:00 — 0.017034
- trace.from_ts-security-service.duration_p95 — 2024-01-24 20:16:00 — 0.025268
- trace.from_ts-inside-payment-service.duration_mean — 2024-01-24 20:19:00 — 0.012118
- trace.from_ts-inside-payment-service.duration_p95 — 2024-01-24 20:19:00 — 0.044096
- trace.from_ts-travel-service.duration_p95 — 2024-01-24 20:23:00 — 0.021634
- trace.from_ts-preserve-service.row_count — 2024-01-24 20:24:00 — 1.0
- trace.from_ts-security-service.row_count — 2024-01-24 20:24:00 — 3.0
- trace.from_ts-travel-service.row_count — 2024-01-24 20:25:00 — 30.0
- trace.from_ts-order-service.row_count — 2024-01-24 20:25:00 — 286.0

Interpretation / next steps
- The strongest signal is concentrated toward ts-order-service (most anomalous trace rows) and ts-travel-service, with a mix of traffic drops and some duration spikes. Investigate upstream traffic routing or throttling and inspect the specific call paths to ts-order-service and ts-travel-service (inside-payment, security, travel, preserve) for root causes of both reduced row_count and increased durations.

The original code execution output of IPython Kernel is also provided below for reference:

(   destination_service  anomaly_count earliest_anomaly_timestamp  example_min_row_count  example_max_duration  mean_severity_ratio
9     ts-order-service             48  2024-01-24 20:16:00+00:00               1.000000              0.855893             0.202056
12   ts-travel-service             38  2024-01-24 20:22:00+00:00               1.000000              2.782713             0.190817
11    ts-train-service             24  2024-01-24 20:25:00+00:00               5.000000              0.233831             0.200492
10    ts-route-service             24  2024-01-24 20:16:00+00:00               6.000000              0.004826             0.166803
1        order-service             12  2024-01-24 20:23:00+00:00               3.000000              0.035929             0.189612
8      ts-auth-service             12  2024-01-24 20:21:00+00:00               5.000000              1.294993             0.252648
6        train-service             12  2024-01-24 20:25:00+00:00              19.000000              0.233831             0.192278
2        route-service             12  2024-01-24 20:16:00+00:00              22.000000              0.004826             0.171685
0         auth-service              6  2024-01-24 20:32:00+00:00              20.000000              0.015942             0.303710
4      station-service              6  2024-01-24 20:36:00+00:00               1.000000              0.835318             0.131879
3         seat-service              6  2024-01-24 20:25:00+00:00               6.000000              0.466537             0.168633
7       travel-service              6  2024-01-24 20:25:00+00:00              95.000000              0.212599             0.165272
5   ticketinfo-service              6  2024-01-24 20:25:00+00:00              14.000000              0.126008             0.281469,                                              trace_name                 timestamp      value
19       trace.from_ts-inside-payment-service.row_count 2024-01-24 20:16:00+00:00   1.000000
32         trace.from_ts-security-service.duration_mean 2024-01-24 20:16:00+00:00   0.017034
33          trace.from_ts-security-service.duration_p95 2024-01-24 20:16:00+00:00   0.025268
424   trace.from_ts-inside-payment-service.duration_... 2024-01-24 20:19:00+00:00   0.012118
425   trace.from_ts-inside-payment-service.duration_p95 2024-01-24 20:19:00+00:00   0.044096
989           trace.from_ts-travel-service.duration_p95 2024-01-24 20:23:00+00:00   0.021634
1115           trace.from_ts-preserve-service.row_count 2024-01-24 20:24:00+00:00   1.000000
1123           trace.from_ts-security-service.row_count 2024-01-24 20:24:00+00:00   3.000000
1259             trace.from_ts-travel-service.row_count 2024-01-24 20:25:00+00:00  30.000000
1247              trace.from_ts-order-service.row_count 2024-01-24 20:25:00+00:00 286.000000)```
```